In [10]:
import pandas as pd
from sqlalchemy import create_engine, text
from neo4j import GraphDatabase

In [14]:
PG_USER = "acatlan"
PG_PASS = "8g4izJPX"
PG_DB = "db_acatlan"
PG_HOST = "172.18.0.5"
PG_PORT = "5432"
SCHEMA = "public"

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:5432/{PG_DB}")

driver = GraphDatabase.driver("bolt://neo4j:7687", auth=None)
 

# Ejemplo Basico

In [15]:
df = pd.DataFrame({
    "cliente":  ["Ana", "Ana", "Luis", "María"],
    "producto": ["Laptop", "Mouse", "Laptop", "Teclado"],
    "monto":    [1200, 25, 1150, 80],
})

CYPHER = """
    MERGE (c:Cliente {nombre: $cliente})
    MERGE (p:Producto {nombre: $producto})
    MERGE (c)-[r:COMPRO]->(p)
    SET r.monto = $monto
"""

with driver.session() as session:
    for fila in df.to_dict("records"):
        session.run(CYPHER, **fila)

Listo


# Ejemplo Elaborado

In [16]:

QUERY = """
SELECT o.order_id, o.order_status, o.customer_id,
       c.customer_city, c.customer_state,
       i.order_item_id, i.price, i.product_id, i.seller_id,
       p.product_category_name, s.seller_city
FROM orders o
JOIN customers c   ON c.customer_id = o.customer_id
JOIN order_items i ON i.order_id    = o.order_id
JOIN products p    ON p.product_id  = i.product_id
JOIN sellers s     ON s.seller_id   = i.seller_id
WHERE o.order_id IN (SELECT order_id FROM orders LIMIT 15)
"""
df = pd.read_sql(text(QUERY), engine)
df["price"] = df["price"].astype(float)
df = df.astype(object).where(df.notna(), None) 

CYPHER = """
MERGE (c:Customer {customer_id: $customer_id})
  SET c.city = $customer_city, c.state = $customer_state
MERGE (o:Order {order_id: $order_id})
  SET o.status = $order_status
MERGE (p:Product {product_id: $product_id})
  SET p.category = $product_category_name
MERGE (s:Seller {seller_id: $seller_id})
  SET s.city = $seller_city
  
MERGE (c)-[:REALIZA]->(o)
MERGE (o)-[r:CONTIENE {item: $order_item_id}]->(p)
  SET r.price = $price
  
MERGE (s)-[:VENDE]->(p)
"""
 
with driver.session() as session:
    for fila in df.to_dict("records"):
        session.run(CYPHER, **fila)